## Libraries

In [0]:
import os
import mlflow
import mlflow.spark

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

## Modeling

In [0]:
# Set MLflow temp directory
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlflow_tmp"

# ----------------------------------------
# Log the preprocessing model (if needed)
# ----------------------------------------
mlflow.spark.log_model(
    model,
    artifact_path="preprocessing_model",
    dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp",
    pip_requirements=["pyspark==4.0.0"]
)

# ----------------------------------------
# Train/test split
# ----------------------------------------
train, test = clean_data.randomSplit([0.8, 0.2], seed=42)

# ----------------------------------------
# Random Forest Model
# ----------------------------------------
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=8
)

evaluator = BinaryClassificationEvaluator(labelCol="label")

# ----------------------------------------
# MLflow experiment tracking
# ----------------------------------------
if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="rf_model"):
    model = rf.fit(train)
    pred = model.transform(test)
    
    auc = evaluator.evaluate(pred)
    mlflow.log_metric("auc", auc)
    
    mlflow.spark.log_model(
        model,
        artifact_path="rf_model",
        input_example=test.limit(1).toPandas(),
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

    print("AUC:", auc)
